# VCM_1_0 — Theory

**Date:** 2026-01-20

This notebook collects the **core equations and definitions** used by the current VCM implementation.
Equations are numbered using `\tag{...}` for easy reference in discussions and manuscripts.


## 1. Dislocation-density representation and jump

For each polyline panel $k$, the solver returns densities $b_I^k$, $b_{II}^k$,
local tangent $\mathbf{t}_k$, normal $\mathbf{n}_k$, and panel weight $\Delta s_k$.

The incremental displacement jump is

$$
\Delta \mathbf{J}_k
= \left( b_{II}^k\, \mathbf{t}_k + b_I^k\, \mathbf{n}_k \right)\, \Delta s_k .
\tag{1}
$$

The accumulated jump $\mathbf{J}(s)$ yields:

- **COD (opening):** $\mathrm{COD} = \mathbf{J}\cdot\mathbf{n}$  
- **CSD (sliding):** $\mathrm{CSD} = \mathbf{J}\cdot\mathbf{t}$


#### 2. Dipolar (full) vs monopolar (half) constraints

### Dipolar (full) neutrality
In the dipolar formulation, neutrality constraints enforce zero net Burgers content per connected crack set:

$$
\int_{\Gamma} b_I(s)\,ds = 0, \qquad \int_{\Gamma} b_{II}(s)\,ds = 0 .
\tag{2}
$$

### Monopolar (half) junction compatibility and gauge
In the monopolar formulation, branches may carry net Burgers content; compatibility is enforced through junction DOFs and constraint rows in the KKT system.


## 3. Stress and displacement fields (superposition)

Let panel midpoint location be $\mathbf{x}_k$ and Burgers increment

$$
\Delta \mathbf{B}_k
= \left( b_{II}^k\, \mathbf{t}_k + b_I^k\, \mathbf{n}_k \right)\, \Delta s_k .
\tag{3}
$$

The global stress field is obtained by superposition of edge-dislocation kernels:

$$
\boldsymbol{\sigma}(\mathbf{x})
= \sum_k \boldsymbol{\sigma}^{\mathrm{edge}}\!\left(\mathbf{x}-\mathbf{x}_k,\, \Delta\mathbf{B}_k\right)
+ \boldsymbol{\sigma}^{\infty}.
\tag{4}
$$

The global displacement field is similarly obtained from the displacement kernel:

$$
\mathbf{u}(\mathbf{x})
= \sum_k \mathbf{u}^{\mathrm{edge}}\!\left(\mathbf{x}-\mathbf{x}_k,\, \Delta\mathbf{B}_k\right)
+ \mathbf{u}^{\infty}.
\tag{5}
$$


## 4. KKT constrained least squares structure

The solver enforces constraints via a KKT system for a least-squares objective. In block form:

$$
\begin{bmatrix}
K^T K & C^T \\
C & 0
\end{bmatrix}
\begin{bmatrix}
q \\ \lambda
\end{bmatrix}
=
\begin{bmatrix}
K^T\,\mathrm{rhs} \\ 0
\end{bmatrix}.
\tag{6}
$$

Here, $q$ are the primary unknown coefficients (densities and junction DOFs, depending on mode),
and $\lambda$ are the Lagrange multipliers enforcing constraints.


## 5. Peach–Koehler (configurational) force density

A PK-like force density in 2D (line direction $\mathbf{e}_z$) is:

$$
\mathbf{f}_{PK}
= \left( \boldsymbol{\sigma}\cdot\mathbf{b} \right)\times\mathbf{e}_z .
\tag{7}
$$

This quantity is stress-based and should not be conflated with the kinematic jump $\mathbf{J}(s)$ or with integrated-density “B-content”.


## 6. Stress intensity factor (SIF) extraction from COD/CSD

Near a crack tip, COD and CSD follow square-root scaling:

$$
\mathrm{COD}(r) \approx A_I \sqrt{\frac{r}{2\pi}}, \qquad
\mathrm{CSD}(r) \approx A_{II} \sqrt{\frac{r}{2\pi}} .
\tag{8}
$$

Fitting the amplitudes gives:

$$
K_I = \frac{\mu}{\kappa+1} A_I, \qquad
K_{II} = \frac{\mu}{\kappa+1} A_{II}.
\tag{9}
$$

with $\kappa = (3-\nu)/(1+\nu)$ (plane stress) or $\kappa = 3-4\nu$ (plane strain).


## 7. Notebook bootstrap (recommended)

In [ ]:
import os, sys
from pathlib import Path

cwd = Path(os.getcwd()).resolve()
root = next((p for p in [cwd] + list(cwd.parents) if (p / "preamble.py").exists()), None)
if root is None:
    raise RuntimeError("preamble.py not found in current directory or any parent")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print("Repo root:", root)



## Galerkin vs Least Square Minimization

In your context:

* **Galerkin method**
  → enforce stationarity of the weak form
  → test functions = trial functions
  → exact satisfaction of constraints in the weak sense

* **Least-squares (LS) minimization**
  → minimize a global residual norm
  → equations satisfied in a weighted ( L^2 ) sense
  → residuals, constraints, and physics can be blended

Your current VCM already *looks* variational, but the moment you add:

* junction constraints,
* gauge fixing,
* multi-branch compatibility,
* PK-based growth criteria,

you are already operating in a regime where **pure Galerkin optimality is no longer sacred**.

---

### 1. Core mathematical difference (in one equation)

#### Galerkin

Solve
[
\delta \Pi(\mathbf{b}) = 0
\quad \text{in the test space}
]

#### Least squares

Minimize
$$
\mathcal{J}(\mathbf{b}) =
|\mathcal{R}_{\text{eq}}(\mathbf{b})|^2

* \alpha |\mathcal{R}_{\text{junction}}(\mathbf{b})|^2
* \beta |\mathcal{R}_{\text{gauge}}(\mathbf{b})|^2
* \cdots
  $$

This difference is subtle but profound.

---

### 2. Advantages of least squares for *your* VCM

#### 2.1 Robustness in large crack networks

**Galerkin weaknesses**

* Junction constraints introduce saddle-point (KKT) systems
* Conditioning degrades rapidly as the number of branches grows
* Local singularities propagate globally through the weak form

**Least-squares advantages**

* Produces **symmetric positive definite (SPD)** systems
* Junctions do not create indefinite blocks
* Conditioning improves with mesh refinement
* Easy to scale to hundreds or thousands of crack segments

📌 **This is a decisive advantage for large crack networks.**

---

#### 2.2 Natural handling of over-constraints and inconsistencies

In realistic fracture problems:

* geometry updates are imperfect,
* intersections are approximate,
* remeshing introduces noise,
* experimental boundary conditions are inconsistent.

**Galerkin**

* Either fails (infeasible constraints)
* Or produces non-physical oscillations

**Least squares**

* Gracefully distributes inconsistency
* Produces the *best possible compromise solution*
* Prevents catastrophic solver breakdown

📌 This is extremely valuable for **crack propagation with intersection**.

---

#### 2.3 Better behavior under dynamic fracture

Dynamic fracture introduces:

* inertia,
* rate-dependent traction,
* transient incompatibilities.

**Galerkin**

* Requires careful time integration to preserve stability
* Sensitive to constraint drift
* Hard to stabilize without artificial damping

**Least squares**

* Allows built-in numerical damping
* Naturally dissipative (without violating physics)
* Stable even with large time steps

📌 LS is better aligned with **implicit dynamic fracture**.

---

#### 2.4 Crack–boundary and crack–interface interactions

When cracks approach:

* free boundaries,
* rigid inclusions,
* material interfaces,

you get **near-singular boundary layers**.

**Galerkin**

* Sensitive to mesh alignment
* Requires special enrichment or singular basis functions

**Least squares**

* Tolerates poor resolution
* Can weight boundary residuals higher
* Does not require exact traction cancellation pointwise

📌 This is a major advantage for **crack–boundary interaction** problems.

---

### 3. Relationship to the PK-window success

Your PK-window method works because:

* it integrates stresses over a neighborhood,
* it suppresses local oscillations,
* it extracts physically meaningful averages.

This is **philosophically identical** to least-squares thinking.

If you move VCM toward LS:

* PK forces become more stable
* Mixed-mode SIFs converge faster
* Crack kinking angles stabilize earlier

You already observed this empirically.

---

### 4. When Galerkin is still preferable

Galerkin is still ideal when:

* single crack,
* clean geometry,
* static loading,
* analytical benchmarks,
* you want exact weak satisfaction.

For example:

* validating kernels,
* convergence studies,
* textbook LEFM cases.

So the right conclusion is **not to replace Galerkin**, but to **add LS as an option**.

---

### 5. Recommended hybrid strategy (best of both worlds)

For VCM, the optimal architecture is:

#### Step 1: Galerkin solve

* Obtain a high-accuracy base solution
* Identify dominant modes

#### Step 2: LS refinement

* Stabilize junctions
* Smooth residuals
* Prepare for propagation

Or even better:

$$
\mathcal{J}
|\delta \Pi|^2

* \lambda |\text{junction mismatch}|^2
  $$

This gives you:

* variational consistency,
* LS robustness,
* scalable performance.

---

### 6. Summary table

| Aspect                    | Galerkin         | Least Squares       |
| ------------------------- | ---------------- | ------------------- |
| Mathematical nature       | Saddle-point     | SPD                 |
| Junction handling         | Fragile          | Robust              |
| Large crack networks      | Poor scaling     | Excellent           |
| Dynamic fracture          | Sensitive        | Stable              |
| Boundary interactions     | Needs enrichment | Tolerant            |
| Crack growth              | Sharp but noisy  | Smooth and reliable |
| Implementation complexity | Moderate         | Moderate            |
| Physical interpretability | Very high        | High                |

---

### 7. Bottom line (very important)

> **For the VCM, least-squares minimization is not a numerical convenience—it is an enabling technology for large-scale, evolving crack networks, dynamics, and realistic geometry.**

Your architecture is already 80% compatible with LS. Introducing it as an optional backend would significantly extend the reach of your method without sacrificing its theoretical foundation.

If you want, next we can:

* derive the LS functional explicitly from your KKT system,
* design a switchable Galerkin/LS solver,
* or analyze conditioning and complexity scaling for large networks.


This is an excellent question, and the short answer is:

> **Because your junction model changes only the *kinematic gauge* (how jumps are distributed), not the *physical traction field* that controls stresses, SIFs, PK forces, and energy release.**

Below I’ll make that precise and connect it directly to your VCM formulation.

---

## 1) What actually determines stresses, SIFs, and PK forces in your VCM

In your formulation, the physical fields are determined by:

* the **distributed Burgers density** along each branch
  [
  \boldsymbol{\beta}(s) = b_{II}(s),\mathbf{t}(s) + b_I(s),\mathbf{n}(s),
  ]
* and the **elastic kernels** mapping this density to stress:
  [
  \boldsymbol{\sigma}(\mathbf{x})
  ===============================

  \int_{\Gamma} \mathsf{K}(\mathbf{x},\mathbf{x}'),\boldsymbol{\beta}(s'),ds'

  * \boldsymbol{\sigma}^{\text{remote}}.
    ]

Everything you compute afterward:

* stress fields,
* stress intensity factors,
* Peach–Koehler forces,
* energy release rates,

depends **only on (\boldsymbol{\beta}(s))** (and the applied stress), *not* on how the jump vector (\mathbf{J}) is distributed at endpoints.

---

## 2) What the junction model *actually* changes

The junction model only affects:

* how the **endpoint jump** (\mathbf{J}) is represented,
* and how it is constrained at vertices.

Recall the exact identity you use everywhere:

[
\mathbf{J}_{\text{end}}
=======================

## \mathbf{J}_{\text{start}}

\int_{\Gamma_{\text{branch}}} \boldsymbol{\beta}(s),ds.
]

Key point:

* (\mathbf{J}) is an **integral of (\boldsymbol{\beta})** plus a gauge choice.
* Changing the junction model **does not change (\boldsymbol{\beta}(s))**.
* It only changes how the *constant of integration* is assigned.

That constant is a **gauge DOF**, not a physical one.

---

## 3) Why the stress field is invariant

Your stress kernels depend on **gradients** of displacement, not absolute displacement jumps.

* Adding a constant jump (\mathbf{J}_0) at a junction:

  * shifts the displacement field by a rigid offset on one side of the crack,
  * but does **not** change strain,
  * therefore does **not** change stress.

So:

* **Strict model**: forces all branches to share the same gauge at the junction
* **Core model**: allows different gauges per branch
* **Soft model**: weakly couples those gauges

But all three produce the **same strain and stress field**, up to solver tolerance.

This is exactly analogous to:

* changing the reference potential in electrostatics,
* or adding a constant to a displacement field in linear elasticity.

---

## 4) Why SIFs and PK forces are unaffected

### Stress Intensity Factors

Your SIF extraction (tip slope, PK-window, etc.) uses:

* near-tip stress fields,
* or energy release rates.

Those depend on:

* (\boldsymbol{\beta}(s)) near the tip,
* not on (\mathbf{J}) at the other end of the branch.

Since:

* tip closure is enforced in all models,
* and (\boldsymbol{\beta}(s)) near the tip is unchanged,

you get the same (K_I, K_{II}).

---

### Peach–Koehler forces

Your PK force is computed as:
[
\mathbf{F}_{PK}
===============

\int_{\text{window}}
(\boldsymbol{\sigma}\cdot d\mathbf{B}) \times \mathbf{e}_z.
]

Again:

* depends on (\boldsymbol{\sigma}),
* and on **local Burgers content** (d\mathbf{B}),
* not on the global endpoint jump gauge.

So the PK force is invariant under junction model changes.

---

## 5) Why this is *good* (and physically correct)

What you’ve discovered is actually a **strength of your formulation**:

* You have cleanly separated:

  * **physical DOFs** → Burgers density (\boldsymbol{\beta}),
  * **gauge DOFs** → endpoint jumps (\mathbf{J}).
* Junction models only manipulate gauge.
* Physical observables are gauge-invariant.

This is exactly how a well-posed variational formulation *should* behave.

---

## 6) When *would* the junction model matter physically?

The junction model would start to affect stresses if you introduced physics that **directly couples to (\mathbf{J})**, for example:

1. **Cohesive-zone or junction-core energy**
   [
   \Pi_{\text{core}} \sim \sum_\alpha |\mathbf{J}_\alpha(v)|^2
   ]
2. **Frictional contact or interlocking at junctions**
3. **Finite junction process zone with traction–separation law**
4. **Nonlinear or finite-strain kinematics**

In those cases:

* (\mathbf{J}) ceases to be a pure gauge,
* and the junction model would change stresses and SIFs.

But in linear elastic fracture with displacement jumps only appearing via their gradients, it should not.

---

## 7) One-sentence takeaway

> **Your junction models change how displacement jumps are *bookkept*, not how elastic fields are generated — so all stress-based quantities remain invariant, as they should.**

This is a strong internal consistency check for your VCM.


For your solver (KKT constraints, mixed parametrizations, junctions, half-cracks), **an active-set method is the better primary approach** for enforcing **COD ≥ 0**. A penalty method is attractive for speed and simplicity, but it will fight your conditioning and can contaminate SIF/PK extraction unless tuned very carefully.

Below is a clear comparison and a recommended implementation path that matches your architecture.

---

## 1) What you are enforcing

You want a unilateral contact constraint on the normal opening:
[
\text{COD}(s) \equiv J(s)\cdot n(s) ;\ge; 0 \quad \text{for all collocation/midpoints}
]

In discrete form at a set of points (i=1..m):
[
g_i(u) \equiv \text{COD}_i(u) \ge 0
]

Your unknowns are densities (or equivalent DOFs) (u), and COD is a **linear map** of those unknowns once the geometry/frame is fixed:
[
\text{COD} = G u + c
]
(you already reconstruct (J) from (b_I, b_{II}) and have (n(s)); this is a linear operator in your discretization).

That’s why this is a standard **linear complementarity / inequality-constrained least squares** problem.

---

## 2) Penalty method vs active set

### A) Penalty method

Enforce by adding a term to the objective, e.g.
[
\min_u |Ku-f|^2 + \rho \sum_{i=1}^m \big(\min(0,, g_i(u))\big)^2
]

**Pros**

* Very easy to implement (no inequality KKT infrastructure)
* No combinatorial “active/inactive” decisions
* Works with your current LS setup (just modify normal equations)

**Cons**

* You must pick penalty weight (\rho); too small → interpenetration persists; too large → severe ill-conditioning
* Penalty contaminates the solution even when COD should be positive (it modifies the physics)
* For crack closure/opening transitions, penalty can produce “soft contact” and nonphysical small penetrations
* With large DOF networks and junction constraints, the system becomes much harder to solve robustly

**Bottom line:** good for quick prototyping and diagnostics, risky for production-quality SIF/PK accuracy.

---

### B) Active set method (inequality enforced exactly)

Solve the constrained problem:
[
\min_u |Ku-f|^2 \quad \text{s.t.}\quad C u = 0,;; g(u)\ge 0
]

Active set turns this into a sequence of **equality-constrained KKT solves** by guessing which constraints are active ((g_i=0)):

* Active set (A): enforce (g_i(u)=0) for (i\in A)
* Inactive set (I): enforce (g_i(u)>0) implicitly

and check complementarity using multipliers (\lambda_i):
[
\lambda_i \ge 0,\quad g_i \ge 0,\quad \lambda_i g_i = 0
]

**Pros**

* Enforces COD ≥ 0 exactly (up to tolerance)
* No penalty parameter tuning
* Clean contact behavior (open where it should open, closed where it should close)
* Typically better conditioning than “huge penalty”
* Produces interpretable contact tractions via multipliers

**Cons**

* More code and logic (iteration + set updates)
* Requires a stable way to build the COD operator (G)
* Can “chatter” if tolerance is not handled carefully (toggle in/out)
* Needs careful handling near tips/junctions (where COD naturally tends to 0)

**Bottom line:** best long-term choice for your solver.

---

## 3) Recommendation for your specific solver

### Use an active set method, with a small optional penalty as a fallback stabilizer.

Reasoning:

* You already run KKT with hard constraints; adding more rows is natural.
* You have robust reconstruction of (J(s)) and frames (n(s)) at midpoints.
* Your goal includes closure diagnostics and physically meaningful contact regions.
* You care about SIF/PK: penalty will distort these unless tuned per case.

---

## 4) Practical implementation details (how to do it cleanly)

### Step 1 — Define discrete COD constraints

Pick the points at which you enforce COD ≥ 0:

* midpoints per panel is a good start (you already compute these)
* optionally, include additional points near tips if needed

For each point (i):
[
g_i(u) = n_i^T J_i(u) \ge 0
]

Since (J) is linear in the density DOFs, construct:
[
g(u) = G u + c
]

You can build (G) by:

* applying the same reconstruction operator you already use for plotting/processor, but in matrix form; or
* (simpler first) compute COD from a candidate solution and build constraints incrementally using Jacobian-free updates (not as clean).

Given your codebase, the right approach is to build a **COD extraction matrix** once per solve:

* `J = R u` (reconstruction operator)
* `COD = N · J` (dot with normal)
  So:
* `G = N R`

### Step 2 — Active set loop

Pseudo-logic:

1. Solve base constrained LS:
   [
   \min |Ku-f|^2 \ \text{s.t.}\ Cu=0
   ]
   get (u)

2. Compute (g=COD(u))

3. Active set update:

* Start with (A = { i : g_i < -\tau })
* Solve with additional equality constraints:
  [
  Cu=0,;; g_A(u)=0
  ]
  i.e. augment (C) with (G_A) rows and RHS (-c_A)

4. Compute multipliers for the active constraints (from KKT solve)

* Remove constraints with (\lambda_i < -\tau_\lambda) (violating (\lambda\ge 0))

Repeat until:

* (g_i \ge -\tau) for all i
* (\lambda_A \ge -\tau_\lambda)

This is the classic active set for convex QP / constrained LS.

### Step 3 — Tip and junction handling

You should not enforce COD ≥ 0 at:

* the exact tip point where COD is forced to zero anyway (or tends to zero)
* junction points where constraints already control jump behavior

Use midpoints only, or explicitly exclude indices that correspond to endpoints/junction neighborhood.

---

## 5) A good compromise approach

If you want the simplest robust path:

1. Implement **active set on midpoints only**
2. Use a small ridge regularization already in your KKT
3. Add a “no-chatter” tolerance band:

   * add-to-active if COD < -1e-10
   * remove-from-active if multiplier < -1e-10
   * otherwise keep membership fixed

This makes it very stable.

---

## 6) When penalty is “better”

Penalty can be preferable if:

* you only need “mostly nonnegative COD” for visualization
* you want a quick stabilizer while debugging geometry/junctions
* you are using iterative solvers and can’t afford repeated KKT solves

But for your stated goal (“add COD>0 constraint to the KKT system”), the exact enforcement property matters, so active set is the right default.

---

## Final recommendation

**Use the active-set method as the primary mechanism.**
Optionally include a small penalty term as a fallback (or as an initial warm start), but don’t rely on penalty alone for production runs.

If you want, next I can propose the exact data structures and modifications to:

* your `constraints.py` (build `G` rows for COD),
* your `KKT.py` (return multipliers cleanly),
* and your `parametrization_v*` solve loop (active-set iterations).
